### The data
There are three data scources used to construct this project, scraped from Sofascore.  
- player_stats_25/26 which contains detailed player level metrics for all players appearing in the Premier League across the 2025/26 season.
- team_stats_25/26 which contains detailed team level metrics for the 2025/26 season.
- player_details which contains basic player information for all players appearing in the Premier league across the 2025/26 season. This extra data sheet was needed for its detailed player position information.

Before proceeding with the analysis we will clean the data to create one player level data sheet with everything needed.

In [1]:
# Import pandas to handle and process the data
import pandas as pd

# Import the three data sources
player_stats = pd.read_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/input/player_stats_25-26.csv")
team_stats = pd.read_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/input/team_stats_25-26.csv")
player_details = pd.read_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/input/player_details.csv")

In [2]:
# 1: Check for duplicate player entries in the data
if not (player_stats["player id"].duplicated().any()): 
    print("No duplicate IDs in player_stats") 
if not (player_details["id"].duplicated().any()):
    print("No duplicate IDs in player_details")

# Check that the player id in player_stats matches the id in player_details
if (player_stats["player id"].isin(player_details["id"]).all()):
    print("All IDs in player_stats have a corresponding ID in player_details")

# Check the number of unique ids in both data sets
print("Players in player_stats: ", player_stats["player id"].nunique())
print("Players in player_details: ", player_details["id"].nunique())

No duplicate IDs in player_stats
No duplicate IDs in player_details
All IDs in player_stats have a corresponding ID in player_details
Players in player_stats:  537
Players in player_details:  537


In [5]:
# 2: Merge player_stats and player_details on player id 
players_df = player_stats.merge(player_details, 
                                how = "left",
                                left_on = "player id",
                                right_on = "id")
# Check it merged correctly
print(players_df[["player","player id","name", "id"]].head())

            player  player id             name       id
0  Bruno Fernandes     288205  Bruno Fernandes   288205
1        Max Weiss    1128801        Max Weiss  1128801
2      Declan Rice     856714      Declan Rice   856714
3  Bruno Guimarães     866469  Bruno Guimarães   866469
4            Rodri     827606            Rodri   827606


In [7]:
# 3: Add the team posession stats to players_df
# Check every player's team ID exists in team_stats
if (players_df["team id"].isin(team_stats["teamId"]).all()):
    print("All teams are accounted for")

# Separate team posession
team_possession = team_stats[["teamId","averageBallPossession"]]

# Round the average possession figures
team_possession["averageBallPossession"] = team_possession["averageBallPossession"].round(2)

# Merge team possession into players_df
players_df = players_df.merge(team_possession,
                             how = "left",
                             left_on = "team id",
                             right_on = "teamId")

# Check it merged correctly
print(players_df[["team id","averageBallPossession"]].head())

All teams are accounted for
   team id  averageBallPossession
0       35                  51.79
1        6                  42.29
2       42                  56.13
3       39                  52.61
4       17                  60.50


In [42]:
# 4: Handle positions, dropping GKs and CBs
# Drop the single row without a detailed position (a Wolves u-21 player)
players_df = players_df.dropna(subset = ["positions_detailed"]).copy()

# Check the data type
print(players_df["positions_detailed"].apply(type).value_counts())

# Modify to a list
import ast
players_df["positions_detailed"] = players_df["positions_detailed"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
print(players_df["positions_detailed"].apply(type).value_counts())

# Find the detailed primary position
players_df["primaryPosition"] = players_df["positions_detailed"].apply(lambda x: x[0])
# Define a position map to group players by position
position_map = {
    "ST": "Striker",
    "LW": "Winger",
    "RW": "Winger",
    "AM": "Midfielder",
    "CM": "Midfielder",
    "DM": "Midfielder",
    "MC": "Midfielder",
    "DL": "Fullback",
    "DR": "Fullback",
    "MR": "Fullback",
    "ML": "Fullback",
    "DC": "Centerback"}

players_df["positionGroup"] = players_df["primaryPosition"].map(position_map)

# Drop goalkeeprs
players_df = players_df[players_df["primaryPosition"] != "GK"].copy()

positions_detailed
<class 'list'>    496
Name: count, dtype: int64
positions_detailed
<class 'list'>    496
Name: count, dtype: int64


In [44]:
# 5: Derive unsuccessful dribbles 

import numpy as np
# Calculate the total dribbles 
players_df["totalDribbles"] = np.where(
    players_df["successfulDribblesPercentage"] > 0,
    players_df["successfulDribbles"] / (players_df["successfulDribblesPercentage"] / 100),
    0)

# Derive the unsuccessful dribbles and ensure integer values
players_df["unsuccessfulDribbles"] = (
    players_df["totalDribbles"] - players_df["successfulDribbles"]).round().astype("Int64")

In [54]:
# 6: Create a final dataframe with only essential metrics and export it
# Define the required columns
feature_cols = ["name","id","team","team id","primaryPosition","positionGroup","minutesPlayed","averageBallPossession","totalShots",
             "inaccuratePasses", "dispossessed","unsuccessfulDribbles","keyPasses","assists","expectedAssists","goals","expectedGoals",
             "market_value"]

# Create the final df
usage_df = players_df[feature_cols].copy()

# Fill nan values with zero
usage_df["expectedAssists"] = usage_df["expectedAssists"].fillna(0)
usage_df["expectedGoals"] = usage_df["expectedGoals"].fillna(0)

# final checks
print(usage_df.shape)
print(usage_df["id"].duplicated().sum())
print(usage_df.isna().sum())

# Export the final df as a csv file
usage_df.to_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/usage_data.csv", index = False)

(496, 18)
0
name                     0
id                       0
team                     0
team id                  0
primaryPosition          0
positionGroup            0
minutesPlayed            0
averageBallPossession    0
totalShots               0
inaccuratePasses         0
dispossessed             0
unsuccessfulDribbles     0
keyPasses                0
assists                  0
expectedAssists          0
goals                    0
expectedGoals            0
market_value             2
dtype: int64
